# Multi-label Defect Classifier — MobileNetV3-Large

**Classes**: `crack`, `water_damage`, `paint_peeling`, `rust` (mold dropped — too few public images).

**Datasets** (download URLs filled below — adjust if Roboflow versions change):
- BD3 (GitHub, MIT) — smartphone building wall photos, primary source
- building-damage-insurance (Roboflow, MIT) — multi-class, indoor/outdoor
- Heritage Defects (Roboflow, CC BY 4.0) — supplemental for rust/peeling

**Output**: `mobilenetv3_defect_4class.pt` saved to `/kaggle/working/`.

Run on Kaggle with **GPU T4 x2** accelerator (uses both via `nn.DataParallel`).

## 1. Install + imports

In [ ]:
!pip install -q timm pytorch-grad-cam roboflow

In [ ]:
import os, json, glob, random, time, shutil, csv
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
import timm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| GPUs:', torch.cuda.device_count())
if DEVICE == 'cuda':
    print(' ', torch.cuda.get_device_name(0))

## 2. Config

In [ ]:
CLASSES = ['crack', 'water_damage', 'paint_peeling', 'rust']
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

IMG_SIZE = 384
# Effective global batch — split across both T4s by DataParallel (32 per GPU).
BATCH_SIZE = 64
NUM_WORKERS = 4
EPOCHS = 25
LR = 5e-4   # scaled up for larger global batch
WEIGHT_DECAY = 1e-4
SEED = 42

DATA_ROOT = Path('/kaggle/working/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path('/kaggle/working')

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

## 3. Download datasets

Replace `ROBOFLOW_API_KEY` with yours from app.roboflow.com → Settings → API. The notebook secrets manager works too: `from kaggle_secrets import UserSecretsClient`.

In [ ]:
import os
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', '')  # set in env or Kaggle secret

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# building-damage-insurance — MIT, multi-defect indoor/outdoor
p1 = rf.workspace('kerdawy').project('building-damage-insurance-jemsv-yndvd')
ds1 = p1.version(1).download('multiclass', location=str(DATA_ROOT/'bdi'))

# Heritage Defects — CC BY 4.0, supplemental rust/peeling/dampness
p2 = rf.workspace('defects-dataset').project('heritage-defects-jnka8')
ds2 = p2.version(p2.versions()[0].version).download('multiclass', location=str(DATA_ROOT/'heritage'))

print('Roboflow downloads done.')

In [ ]:
# BD3 — clone from GitHub (MIT, real smartphone wall photos)
!git clone --depth 1 https://github.com/Praveenkottari/BD3-Dataset {DATA_ROOT}/bd3
!ls {DATA_ROOT}/bd3

## 4. Build unified manifest

Each source has its own folder layout and class names. We map everything to our 4-class taxonomy and produce one DataFrame with columns `[path, crack, water_damage, paint_peeling, rust]` (binary multi-hot).

In [ ]:
# Map raw labels (lowercased) to one of our 4 classes (or None to drop)
LABEL_MAP = {
    # crack
    'crack': 'crack', 'major_crack': 'crack', 'minor_crack': 'crack',
    'major crack': 'crack', 'minor crack': 'crack', 'cracks': 'crack',
    # water_damage
    'stain': 'water_damage', 'damp': 'water_damage', 'dampness': 'water_damage',
    'water_seepage': 'water_damage', 'seepage': 'water_damage',
    'efflorescence': 'water_damage', 'dampness with fungus': 'water_damage',
    'water damage': 'water_damage', 'water_damage': 'water_damage',
    # paint_peeling
    'peeling': 'paint_peeling', 'peeling_paint': 'paint_peeling',
    'peeling paint': 'paint_peeling', 'paintpeel': 'paint_peeling',
    'paint_peeling': 'paint_peeling', 'flaking plaster': 'paint_peeling',
    'flaking_plaster': 'paint_peeling',
    # rust
    'rust': 'rust', 'corrosion': 'rust', 'moderate corrosion': 'rust',
    # drop
    'mold': None, 'mould': None, 'algae': None, 'fungus': None,
    'no_defect': None, 'normal': None,
}

def normalize(name):
    return name.strip().lower().replace('-', '_')

def empty_row(path):
    return {'path': str(path), **{c: 0 for c in CLASSES}}

In [ ]:
# --- Roboflow multi-label CSV format: train/_classes.csv with columns [filename, class1, class2, ...] ---
def ingest_roboflow_multilabel(root):
    rows = []
    for split in ['train', 'valid', 'test']:
        csv_path = Path(root) / split / '_classes.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        cols = [c for c in df.columns if c.lower() != 'filename']
        for _, r in df.iterrows():
            img_path = Path(root) / split / r['filename']
            if not img_path.exists():
                continue
            row = empty_row(img_path)
            for c in cols:
                if int(r[c]) == 1:
                    mapped = LABEL_MAP.get(normalize(c))
                    if mapped:
                        row[mapped] = 1
            if any(row[c] for c in CLASSES):
                rows.append(row)
    return rows

# --- BD3 single-class folder format: bd3/<class_name>/img.jpg ---
def ingest_folder_per_class(root):
    rows = []
    root = Path(root)
    for cls_dir in root.glob('**/'):
        if not cls_dir.is_dir():
            continue
        mapped = LABEL_MAP.get(normalize(cls_dir.name))
        if not mapped:
            continue
        for img in cls_dir.glob('*'):
            if img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                row = empty_row(img)
                row[mapped] = 1
                rows.append(row)
    return rows

all_rows = []
all_rows += ingest_roboflow_multilabel(DATA_ROOT/'bdi')
all_rows += ingest_roboflow_multilabel(DATA_ROOT/'heritage')
all_rows += ingest_folder_per_class(DATA_ROOT/'bd3')

manifest = pd.DataFrame(all_rows).drop_duplicates(subset='path').reset_index(drop=True)
print('Total images:', len(manifest))
print('Per-class positives:')
print(manifest[CLASSES].sum())

## 5. Train/val split + dataset

In [ ]:
# Stratified-ish split. For multi-label we just use random with the same seed.
train_df, val_df = train_test_split(manifest, test_size=0.15, random_state=SEED)
print(f'train={len(train_df)} val={len(val_df)}')

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class DefectDataset(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['path']).convert('RGB')
        img = self.tf(img)
        y = torch.tensor([r[c] for c in CLASSES], dtype=torch.float32)
        return img, y

train_ds = DefectDataset(train_df, train_tf)
val_ds = DefectDataset(val_df, val_tf)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

## 6. Model + training

In [ ]:
base_model = timm.create_model('mobilenetv3_large_100', pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)
if torch.cuda.device_count() > 1:
    print(f'Using DataParallel across {torch.cuda.device_count()} GPUs')
    model = nn.DataParallel(base_model)
else:
    model = base_model

def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

# pos_weight handles class imbalance — pushes the model to learn rare classes.
pos_counts = train_df[CLASSES].sum().values
neg_counts = len(train_df) - pos_counts
pos_weight = torch.tensor(neg_counts / np.maximum(pos_counts, 1), dtype=torch.float32).to(DEVICE)
print('pos_weight:', dict(zip(CLASSES, pos_weight.cpu().numpy().round(2))))

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)

In [ ]:
def run_epoch(loader, train=True):
    model.train(train)
    losses, all_y, all_p = [], [], []
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optim.zero_grad(); loss.backward(); optim.step()
        losses.append(loss.item())
        all_y.append(y.cpu().numpy()); all_p.append(torch.sigmoid(logits).detach().cpu().numpy())
    y_arr = np.concatenate(all_y); p_arr = np.concatenate(all_p)
    pred = (p_arr > 0.5).astype(int)
    f1 = f1_score(y_arr, pred, average='macro', zero_division=0)
    aps = [average_precision_score(y_arr[:, i], p_arr[:, i]) if y_arr[:, i].sum() > 0 else float('nan')
           for i in range(NUM_CLASSES)]
    return np.mean(losses), f1, aps

best_f1 = 0
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_f1, _ = run_epoch(train_loader, train=True)
    va_loss, va_f1, va_ap = run_epoch(val_loader, train=False)
    sched.step()
    print(f'ep{epoch:02d}  tr_loss={tr_loss:.3f} f1={tr_f1:.3f} | va_loss={va_loss:.3f} f1={va_f1:.3f} '
          f'AP=' + ' '.join(f'{c}:{ap:.2f}' for c, ap in zip(CLASSES, va_ap)) +
          f'  ({time.time()-t0:.0f}s)')
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save({'state_dict': unwrap(model).state_dict(), 'classes': CLASSES, 'img_size': IMG_SIZE,
                    'mean': MEAN, 'std': STD},
                   OUT_DIR/'mobilenetv3_defect_4class.pt')
        print('  ↳ saved (best)')

## 7. Grad-CAM smoke test

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import matplotlib.pyplot as plt

ckpt = torch.load(OUT_DIR/'mobilenetv3_defect_4class.pt', map_location=DEVICE)

# Handle loading whether the current model is wrapped in DataParallel or not
state_dict = ckpt['state_dict']
if isinstance(model, nn.DataParallel) and not list(state_dict.keys())[0].startswith('module.'):
    # model is DP but state_dict is not
    state_dict = {f'module.{k}': v for k, v in state_dict.items()}
elif not isinstance(model, nn.DataParallel) and list(state_dict.keys())[0].startswith('module.'):
    # state_dict is DP but model is not
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.eval()

# MobileNetV3 last conv block — used as Grad-CAM target.
target_layer = [unwrap(model).blocks[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

# Sample a positive example from val
row = val_df[val_df[CLASSES].sum(axis=1) > 0].iloc[0]
img = Image.open(row['path']).convert('RGB')
x = val_tf(img).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    probs = torch.sigmoid(model(x))[0].cpu().numpy()
top = int(np.argmax(probs))
print('Predicted:', dict(zip(CLASSES, probs.round(2))), '| top:', CLASSES[top])

from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(top)])[0]
rgb = np.array(img.resize((IMG_SIZE, IMG_SIZE))).astype(np.float32) / 255.0
overlay = show_cam_on_image(rgb, heat, use_rgb=True)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(rgb); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(overlay); ax[1].set_title(f'Grad-CAM → {CLASSES[top]}'); ax[1].axis('off')
plt.show()